<a href="https://colab.research.google.com/github/MykolaVashchenko/olx_flats_parser/blob/main/parse.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gspread oauth2client playwright
!playwright install-deps
!playwright install chromium

In [35]:
import asyncio
import json
from playwright.async_api import async_playwright
import gspread
from oauth2client.service_account import ServiceAccountCredentials
from google.colab import userdata

def get_google_sheet(sheet_id):
    scope = ["https://spreadsheets.google.com/feeds", "https://www.googleapis.com/auth/drive"]
    try:
        secret_raw = userdata.get('GCP_JSON_KEY')
        creds_dict = json.loads(secret_raw)
        creds = ServiceAccountCredentials.from_json_keyfile_dict(creds_dict, scope)
        client = gspread.authorize(creds)

        sheet = client.open_by_key(sheet_id)
        return sheet.sheet1
    except Exception as e:
        print(f"Authorization error: {e}")
        return None

async def scrape_olx_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        url = "https://www.olx.ua/uk/nedvizhimost/kvartiry/dolgosrochnaya-arenda-kvartir/kiev/"
        await page.goto(url, wait_until="domcontentloaded")

        try:
            await page.wait_for_selector('div[data-cy="l-card"]', timeout=10000)
        except Exception as e:
            print("No content")

        ads = await page.query_selector_all('div[data-cy="l-card"]')
        total_found = len(ads)
        print(f"Found ads: {total_found}\n")

        scraped_data = []
        valid_count = 0
        garbage_ads = 0

        for index, ad in enumerate(ads, 1):
            try:
                price_text = await ad.query_selector('p[data-testid="ad-price"]')
                if not price_text:
                    price_text = await ad.query_selector('p[data-cy="ad-price"]')

                if price_text:
                    price = await price_text.text_content()
                else:
                    price = "0"

                clean_price = "".join(filter(str.isdigit, price))

                if not clean_price:
                    clean_price = "0"

                price_int = int(clean_price)

                if price_int > 250000:
                    garbage_ads += 1
                    continue

                if price_int < 1000:
                    garbage_ads += 1
                    continue

                location_el = await ad.query_selector('p[data-testid="location-date"]')

                if location_el:
                    location_text = await location_el.text_content()
                else:
                    location_text = "Unknown"

                location_parts = location_text.split(' - ')
                location_first_part = location_parts[0]
                city_parts = location_first_part.split(',')
                parsed_location = city_parts[0].strip()

                link_el = await ad.query_selector('a')
                if not link_el:
                    garbage_ads += 1
                    continue

                href = await link_el.get_attribute('href')

                if href.startswith('/'):
                    full_link = "https://www.olx.ua" + href
                else:
                    full_link = href

                detail_page = await context.new_page()
                await detail_page.goto(full_link, wait_until="domcontentloaded")

                params = await detail_page.evaluate('''() => {
                    let res = {};
                    res.title = 'Без назви';
                    res.floor = '-';
                    res.total_floors = '-';
                    res.area = '-';

                    let headers = document.querySelectorAll('h1, h4');

                    for (let h of headers) {
                        let text = h.innerText.trim();
                        let textLength = text.length;

                        if (textLength > 10) {
                            if (!text.includes('Повідомлення')) {
                                res.title = text;
                                break;
                            }
                        }
                    }

                    if (res.title === 'Без назви') {
                        let docTitle = document.title;
                        let titleParts = docTitle.split(' - ');
                        res.title = titleParts[0].trim();
                    }

                    let items = document.querySelectorAll('p');

                    for (let p of items) {
                        let text = p.innerText;

                        if (text.includes('Поверховість:')) {
                            let floorTotalText = text.replace('Поверховість:', '');
                            res.total_floors = floorTotalText.trim();
                        }

                        if (text.includes('Поверх:')) {
                            let floorText = text.replace('Поверх:', '');
                            res.floor = floorText.trim();
                        }

                        if (text.includes('Загальна площа:')) {
                            let areaText = text.replace('Загальна площа:', '');
                            let cleanArea = areaText.replace(/[^0-9,.]/g, '');
                            res.area = cleanArea.trim();
                        }
                    }
                    return res;
                }''')

                scraped_data.append([
                    params['title'],
                    clean_price,
                    params['floor'],
                    params['total_floors'],
                    parsed_location,
                    params['area']
                ])

                valid_count += 1

                short_title = params['title'][:30]
                print(f"Added: {short_title}....       Collected: {valid_count}")

                await detail_page.close()

            except Exception as e:
                continue

        print(f"\nFound ads: {total_found}")
        print(f"Successfully collected: {valid_count}")
        print(f"Rejected ads (garbage): {garbage_ads}")

        await browser.close()
        return scraped_data

async def main():
    SPREADSHEET_ID = "1xdXTMFDn3eGboIoHL3Ns6ZaqRypX00OljG4aF150E7k"
    sheet = get_google_sheet(SPREADSHEET_ID)

    if sheet:
        sheet.clear()

        headers = ["Назва", "Ціна", "Поверх", "Поверховість", "Населений пункт", "Площа"]
        sheet.append_row(headers)

        format_settings = {"textFormat": {"bold": True}}
        sheet.format("A1:F1", format_settings)

        data = await scrape_olx_data()

        if data:
            sheet.append_rows(data)
        else:
            print("Data was not found")

await main()

Found ads: 52

Added: Оренда смарт квартири біля мет....       Collected: 1
Added: Оренда 1к квартира 44 м2 ЖК Sv....       Collected: 2
Added: ВЕЛИКА НЕРУХОМІСТЬ....       Collected: 3
Added: Здається 1 кімн. квартира по в....       Collected: 4
Added: Здам квартиру хозяйка Шепелева....       Collected: 5
Added: Без комісії!!! Довгострокова о....       Collected: 6
Added: Здається стильна 1к квартира в....       Collected: 7
Added: 1к (студия), Шевченковский, пр....       Collected: 8
Added: Оренда БЕЗ КОМІСІЇ. Здам 2к кв....       Collected: 9
Added: Власник Довгострокова Оренда 2....       Collected: 10
Added: denisenko218737....       Collected: 11
Added: Оренда нерухомості....       Collected: 12
Added: Видова квартира в ЖК Французьк....       Collected: 13
Added: Костянтинівська 4-кімнатна 2-р....       Collected: 14
Added: Оренда 2-кімнатної квартири пр....       Collected: 15
Added: Здам 1 кімнатну смарт квартиру....       Collected: 16
Added: Власник. Оренда квартири ЖК Ма....